### Download Dependencies

In [ ]:
%pip install kagglehub requests pandas beautifulsoup4

### Setup

In [1]:
OUTPUT_DIR = "dataset"

### Kaggle

In [ ]:
import kagglehub
import os

In [ ]:
def download_dataset(link):
    output_dir = OUTPUT_DIR + "/" + link.split("/")[-1]
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        path = kagglehub.dataset_download(link, output_dir=output_dir)
        print("Path to dataset files:", path)
    else:
        print("Dataset already exists at:", output_dir)

#### BBC

In [ ]:
link = "gpreda/bbc-news"
download_dataset(link)

#### HUFFPOST

In [ ]:
link = "rmisra/news-category-dataset"
download_dataset(link)

### theguardian.com/world

In [ ]:
import requests
import pandas as pd
import time
import os
from datetime import datetime, timedelta

# --- CONFIGURATION ---
try:
    from config import GUARDIAN_API_KEY
except Exception:
    GUARDIAN_API_KEY = os.environ.get('GUARDIAN_API_KEY')
    if not GUARDIAN_API_KEY:
        raise RuntimeError('GUARDIAN_API_KEY not found. Add it to config.py or set GUARDIAN_API_KEY env var.')

START_YEAR = 2016
START_MONTH = 1
MONTHS_TO_SCRAPE = 120 # 10 years
output_dir = OUTPUT_DIR + "/guardian_dataset"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

def get_month_range(year, month):
    start_date = datetime(year, month, 1)
    if month == 12:
        end_date = datetime(year + 1, 1, 1) - timedelta(days=1)
    else:
        end_date = datetime(year, month + 1, 1) - timedelta(days=1)
    return start_date.strftime('%Y-%m-%d'), end_date.strftime('%Y-%m-%d')

def scrape_multiple_months(start_y, start_m, num_months):
    current_y, current_m = start_y, start_m
    
    for _ in range(num_months):
        filename = f"guardian_{current_y}_{current_m:02d}.csv"
        file_path = os.path.join(output_dir, filename)

        if os.path.exists(file_path):
            print(f"Skipping {current_y}-{current_m:02d}... ({file_path} already exists)")
            current_y, current_m = increment_month(current_y, current_m)
            continue

        start_d, end_d = get_month_range(current_y, current_m)
        base_url = "https://content.guardianapis.com/search"
        all_articles = []
        
        params = {
            'api-key': GUARDIAN_API_KEY,
            'from-date': start_d,
            'to-date': end_d,
            'page-size': 200,   # MAXIMIZED: 200 articles per API call
            'show-fields': 'trailText',
            'order-by': 'oldest'
        }

        try:
            first_page = requests.get(base_url, params=params).json()
            total_pages = first_page['response']['pages']
            print(f"\n--- Scraping {start_d} to {end_d} ({total_pages} pages) ---")

            for page in range(1, total_pages + 1):
                params['page'] = page
                
                success = False
                for attempt in range(3): 
                    try:
                        response = requests.get(base_url, params=params, timeout=10)
                        
                        if response.status_code == 429:
                            print("\n[!] HIT DAILY LIMIT! Run the script again tomorrow.")
                            return
                        
                        data = response.json()['response']['results']
                        for art in data:
                            all_articles.append({
                                'Title': art['webTitle'],
                                'Description': art.get('fields', {}).get('trailText', ''),
                                'Date': art['webPublicationDate'],
                                'Link': art['webUrl']
                            })
                        
                        print(f"Finished page {page}/{total_pages}", end='\r')
                        time.sleep(1) 
                        success = True
                        break 
                        
                    except requests.exceptions.RequestException:
                        print(f"\nConnection hiccup on page {page}. Retrying in 5 seconds... (Attempt {attempt+1}/3)")
                        time.sleep(5)
                
                if not success:
                    print(f"\n[!] Failed to get page {page} after 3 attempts. Stopping month.")
                    return 

            df = pd.DataFrame(all_articles)
            df.to_csv(file_path, index=False, encoding='utf-8')
            print(f"\nSaved {file_path} successfully! (Total Articles: {len(df)})")
            
        except Exception as e:
            print(f"\nError during {current_y}-{current_m}: {e}")
            break

        current_y, current_m = increment_month(current_y, current_m)

def increment_month(y, m):
    if m == 12:
        return y + 1, 1
    return y, m + 1

# Start the engine
scrape_multiple_months(START_YEAR, START_MONTH, MONTHS_TO_SCRAPE)

### Wikipedia

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
import calendar
import copy  

# --- CONFIGURATION ---
START_YEAR = 2025
START_MONTH = 1
MONTHS_TO_SCRAPE = 24

HEADERS = {
    'User-Agent': 'NewsDatasetBot/1.0 (Educational Research Project)'
}

output_dir = OUTPUT_DIR + "/wikipedia_dataset"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

def process_list_items(list_items, exact_date, events_list):
    """The highly tuned engine that extracts perfectly formatted rows from <li> tags"""
    for li in list_items:
        # 1. Clone the HTML
        clone = copy.copy(li)
        
        # 2. Prevent overlapping text (remove nested bullets)
        for ul in clone.find_all('ul'):
            ul.decompose()
            
        # 3. Get Description with proper spacing
        desc_text = " ".join(clone.get_text(separator=" ", strip=True).split())
        
        # 4. Get Title (Category)
        title_text = ""
        if li.parent and li.parent.parent and li.parent.parent.name == 'li':
            parent_clone = copy.copy(li.parent.parent)
            for ul in parent_clone.find_all('ul'):
                ul.decompose() 
            title_text = " ".join(parent_clone.get_text(separator=" ", strip=True).split())
        
        # 5. Extract links
        all_links = clone.find_all('a', href=lambda href: href and href.startswith('http'))
        seen_urls = set()
        
        if len(desc_text) > 15:
            for link_tag in all_links:
                url = link_tag['href']
                
                # 6. Filter for unique, external news sources only
                if 'wikipedia.org' not in url and 'wikimedia.org' not in url:
                    if url not in seen_urls:
                        seen_urls.add(url) 
                        
                        events_list.append({
                            'Title': title_text,       
                            'Description': desc_text,  
                            'Date': exact_date,
                            'Link': url
                        })

def scrape_wikipedia_months(start_y, start_m, num_months):
    current_y, current_m = start_y, start_m
    
    for _ in range(num_months):
        filename = f"wiki_news_{current_y}_{current_m:02d}.csv"
        filepath = os.path.join(output_dir, filename)
        # Resume capability: Skip months we already downloaded
        if os.path.exists(filepath):
            print(f"Skipping {current_y}-{current_m:02d}... ({filepath} already exists)")
            current_y, current_m = increment_month(current_y, current_m)
            continue
            
        month_name = calendar.month_name[current_m]
        url = f"https://en.wikipedia.org/wiki/Portal:Current_events/{month_name}_{current_y}"
        
        print(f"\n--- Scraping Online: {month_name} {current_y} ---")
        all_events = []
        
        try:
            # Hit the live Wikipedia server
            response = requests.get(url, headers=HEADERS)
            
            if response.status_code != 200:
                print(f"[!] Failed to load {url} (Status: {response.status_code})")
                current_y, current_m = increment_month(current_y, current_m)
                time.sleep(2)
                continue

            soup = BeautifulSoup(response.text, 'html.parser')
            
            # --- STRATEGY 1: Historical 'vevent' format (2016-era layout) ---
            event_blocks = soup.find_all(class_='vevent')
            
            if event_blocks:
                for block in event_blocks:
                    date_header = block.find(class_='summary')
                    exact_date = date_header.get_text(strip=True) if date_header else f"{month_name} {current_y}"
                    
                    description_cell = block.find(class_='description')
                    if description_cell:
                        process_list_items(description_cell.find_all('li'), exact_date, all_events)
            
            # --- STRATEGY 2: Modern standard list format (Newer layout fallback) ---
            else:
                content_div = soup.find('div', {'class': 'mw-parser-output'})
                if content_div:
                    current_day = f"{month_name} {current_y}" 
                    
                    for child in content_div.children:
                        # Find the date headers
                        if child.name in ['h2', 'h3', 'h4']:
                            headline = child.find(class_='mw-headline')
                            if headline:
                                current_day = headline.get_text(strip=True)
                                
                        # Find the lists under those headers
                        elif child.name == 'ul':
                            process_list_items(child.find_all('li', recursive=False), current_day, all_events)

            # --- SAVE DATA ---
            df = pd.DataFrame(all_events, columns=['Title', 'Description', 'Date', 'Link'])
            if not df.empty:
                df.to_csv(filepath, index=False, encoding='utf-8')
                print(f"Saved {filepath}! (Extracted {len(df)} news events)")
            else:
                print(f"[!] No valid events parsed for {month_name} {current_y}.")

        except Exception as e:
            print(f"Error during {month_name} {current_y}: {e}")
            
        # Be polite to Wikipedia's servers
        time.sleep(2) 
        current_y, current_m = increment_month(current_y, current_m)

def increment_month(y, m):
    if m == 12:
        return y + 1, 1
    return y, m + 1

# Start the engine
scrape_wikipedia_months(START_YEAR, START_MONTH, MONTHS_TO_SCRAPE)

### GDELT 2.0

In [8]:
import os
from datetime import datetime, timedelta
from GDELT_collector import collect_news, save_csv

# --- TRUSTED NEWS SOURCES (Domain Filtering) ---
TIER_1 = ["reuters.com", "apnews.com", "bbc.co.uk", "nytimes.com", "theguardian.com"]
TIER_2 = ["washingtonpost.com", "ft.com", "economist.com", "nature.com", "aljazeera.com"]
TIER_3 = ["dw.com", "channelnewsasia.com", "nhk.or.jp", "southchinamorningpost.com", "politico.com"]
TRUSTED_DOMAINS = TIER_1 + TIER_2 + TIER_3

# --- CONFIGURATION ---
START_YEAR = 2026
START_MONTH = 1
MONTHS_TO_SCRAPE = 5  # 10 years
MIN_MENTIONS = 1
KEYWORDS = None  # Example: ["Ukraine", "Russia", "China"]

output_dir = OUTPUT_DIR + "/gdelt_dataset"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

def month_date_range(year, month):
    start = datetime(year, month, 1)
    if month == 12:
        end = datetime(year + 1, 1, 1) - timedelta(days=1)
    else:
        end = datetime(year, month + 1, 1) - timedelta(days=1)
    return start, end

def increment_month(y, m):
    if m == 12:
        return y + 1, 1
    return y, m + 1

def scrape_gdelt_months(start_y, start_m, num_months):
    current_y, current_m = start_y, start_m

    for _ in range(num_months):
        filename = f"gdelt_{current_y}_{current_m:02d}.csv"
        file_path = os.path.join(output_dir, filename)

        if os.path.exists(file_path):
            print(f"Skipping {current_y}-{current_m:02d}... ({file_path} already exists)")
            current_y, current_m = increment_month(current_y, current_m)
            continue

        start_d, end_d = month_date_range(current_y, current_m)
        print(f"\n--- Scraping GDELT: {start_d.strftime('%Y-%m-%d')} to {end_d.strftime('%Y-%m-%d')} ---")

        try:
            news_items = collect_news(
                start_date=start_d,
                end_date=end_d,
                keywords=KEYWORDS,
                min_mentions=MIN_MENTIONS,
                domains=TRUSTED_DOMAINS,
                head_check=False,
                workers=64,
                batch_size=128,
                batch_delay=0.5
            )
            
            if news_items:
                save_csv(news_items, file_path)
                print(f"Saved {file_path}! (Total Articles: {len(news_items)})")
            else:
                print(f"[!] No articles from trusted sources for {current_y}-{current_m:02d}.")

        except Exception as e:
            print(f"Error during {current_y}-{current_m:02d}: {e}")
            break

        current_y, current_m = increment_month(current_y, current_m)

# Start the engine
scrape_gdelt_months(START_YEAR, START_MONTH, MONTHS_TO_SCRAPE)


--- Scraping GDELT: 2026-01-01 to 2026-01-31 ---
[16:12:00] Time slices: 2976 (2026-01-01 to 2026-01-31)
[16:12:00] Workers: 64, Batch size: 128
[16:12:00] Skipping HEAD check (assuming all slices exist)


[16:12:04]   Progress: 128/2976 | dl: 128 | skip: 0 | news: 1045 | 30.2 slices/s | ETA: 94s
[16:12:09]   Progress: 256/2976 | dl: 256 | skip: 0 | news: 2100 | 28.0 slices/s | ETA: 97s
[16:12:13]   Progress: 384/2976 | dl: 384 | skip: 0 | news: 3229 | 27.7 slices/s | ETA: 94s
[16:12:21]   Progress: 512/2976 | dl: 512 | skip: 0 | news: 5182 | 24.4 slices/s | ETA: 101s
[16:12:28]   Progress: 640/2976 | dl: 640 | skip: 0 | news: 7772 | 22.2 slices/s | ETA: 105s
[16:12:36]   Progress: 768/2976 | dl: 768 | skip: 0 | news: 9793 | 21.1 slices/s | ETA: 105s
[16:12:44]   Progress: 896/2976 | dl: 896 | skip: 0 | news: 11562 | 20.3 slices/s | ETA: 102s
[16:12:49]   Progress: 1024/2976 | dl: 1024 | skip: 0 | news: 12953 | 20.9 slices/s | ETA: 93s
[16:12:55]   Progress: 1152/2976 | dl: 1152 | skip: 0 | news: 14508 | 20.9 slices/s | ETA: 87s
[16:13:02]   Progress: 1280/2976 | dl: 1280 | skip: 0 | news: 16208 | 20.4 slices/s | ETA: 83s
[16:13:11]   Progress: 1408/2976 | dl: 1408 | skip: 0 | news: 1882